In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.insert(0, str(Path(r"C:\Users\taylorhearn\git_repos\vascumap\bel_vascumap")))
from plotting import (combine_outputs, pca_plots, plot_experiment_comparisons,
                       plot_orientation_kde)

# ── Experiment-specific config ────────────────────────────────────────────────
CONDITION_COLORS = {
    "static": "red", "flow": "dodgerblue",
    "high_flow": "mediumblue", "low_flow": "lightblue",
    "resevoir": "darkorange",
}

def find_condition(row, experiment_setup="old"):
    name = str(row["image_name"]).lower()
    if "static" in name:
        return "static"
    elif "resevoir" in name or "reservoir" in name:
        return "resevoir"
    elif experiment_setup == "old":
        return "flow" if "flow" in name else "NA"
    else:
        if any(f"valve {n}" in name for n in [1, 2, 7, 8]):
            return "high_flow"
        elif any(f"valve {n}" in name for n in [3, 4, 5, 6]):
            return "low_flow"
        return "NA"

In [ ]:
root_dir = Path(r"Z:\Bel\Farid_bel\test_old")
output_dir = root_dir.parent / f"{root_dir.name}_output_data"
output_dir.mkdir(parents=True, exist_ok=True)

experiment_setup = "old"  # "old" = two conditions (flow/static); else multiclass
if experiment_setup == "old":
    xorder = ["static", "flow"]
else:
    xorder = ["static", "resevoir", "low_flow", "high_flow"]

combined_analysis_metrics = pd.read_csv(r"z:\Bel\Farid_bel\Old_Experiment_Outputs_output_data\farid_combined_analysis_metrics.csv")
combined_branch_metrics = pd.read_csv(r"z:\Bel\Farid_bel\Old_Experiment_Outputs_output_data\farid_combined_branch_metrics.csv")
print(combined_analysis_metrics.shape)

filenames_to_remove = [
    '18.09.25 flow 6million day 7_img3_Valve 4',
    '30.10.25 flow 6million day 7_img1_Valve 2',
    '30.10.25 flow 6million day 7_img2_Valve 4',
]
combined_analysis_metrics = combined_analysis_metrics[~combined_analysis_metrics["image_name"].isin(filenames_to_remove)]
combined_branch_metrics = combined_branch_metrics[~combined_branch_metrics["image_name"].isin(filenames_to_remove)]
combined_analysis_metrics.to_csv(str(output_dir) + "/FARID_OLD_FINAL_GOOD_ONLY_analysis_metrics.csv")
combined_branch_metrics.to_csv(str(output_dir) + "/FARID_OLD_FINAL_GOOD_ONLY_branch_metrics.csv")
print(combined_analysis_metrics.shape)

(29, 34)
(26, 34)


In [ ]:
COLS_TO_DROP = [
    "median_sprout_and_branch_median_cs_area_um2",
    "median_internal_pore_area_um2",
    "median_internal_pore_max_inscribed_radius_um",
    "p90_minus_p10_internal_pore_area_um2",
    "total_internal_pore_count",
    "p90_minus_p10_internal_pore_max_inscribed_radius_um",
    "internal_pore_area_fraction_in_filled_vascular_area",
]
combined_analysis_metrics_minimal = combined_analysis_metrics.drop(columns=COLS_TO_DROP)

In [ ]:
significant_params = pca_plots(combined_analysis_metrics_minimal, CONDITION_COLORS, save_dir=output_dir)

In [ ]:
plot_experiment_comparisons(combined_analysis_metrics_minimal, significant_params[:5],
                            xorder, "significant_changes", output_dir, CONDITION_COLORS)
plot_orientation_kde(combined_branch_metrics, CONDITION_COLORS, save_dir=output_dir)